# Web Application Vulnerability Scanning, Risk Evaluation & Alert System
**Assignment 3 - Solution Notebook**

## Requirement 1: Vulnerability Scanning System
> Build a scanner that probes a target URL and detects security weaknesses.  
> • Detect at least 5 vulnerability types  
> • Assign each finding a severity level: Critical, High, Medium, Low, or Informational  
> • Handle errors — no crashes on invalid or unreachable URLs

---

### Solution:

Two functions form the scanning engine:

| Function | Role |
|---|---|
| `run_nmap_scan(target)` | Runs `nmap -Pn -sV` on the target and saves results to XML |
| `parse_nmap(xml_file, target)` | Parses the XML and classifies each open port into one of **6 vulnerability types** with severity + score |
| `check_virustotal(target, api_key)` | Queries VirusTotal's domain reputation API — adds a **Critical** finding if flagged |

**The 6+ vulnerability types detected:**

| # | Vulnerability Type | Severity | Score |

| 1 | Cleartext Protocol Exposed (FTP / Telnet) | Critical | 9 |

| 2 | Remote Admin Interface Exposed (SSH / RDP) | High | 7 |

| 3 | Database Exposed to Public (MySQL / PostgreSQL / MSSQL / MongoDB) | High | 8 |

| 4 | Unencrypted Web Traffic (HTTP / Port 80) | Medium | 4 |

| 5 | Generic Open Port (any other service) | Low | 2 |

| 6 | Malicious Domain Reputation (VirusTotal) | Critical | 10 |

**Error handling:** All scan logic is wrapped in `try/except` blocks. If the target is unreachable or the XML is missing, the function returns an `Informational` finding instead of crashing.

In [3]:
#REQUIREMENT 1

import subprocess
import xml.etree.ElementTree as ET
import requests
import os

SCAN_DIR = "scan_results"
os.makedirs(SCAN_DIR, exist_ok=True)

#Scanner: runs nmap and saves XML output 
def run_nmap_scan(target):
    xml_file = f"{SCAN_DIR}/{target}.xml"
    try:
        subprocess.run(
            ["nmap", "-Pn", "-sV", "-oX", xml_file, target],
            capture_output=True, timeout=120
        )
    except Exception as e:
        pass #error handled in parse_nmap below
    return xml_file

# Parser: classifies open ports into vulnerability types 
def parse_nmap(xml_file, target):
    findings = []

    # Error handling for unreachable / invalid target
    if not os.path.exists(xml_file):
        return [{"Target": target, "Vulnerability": "Host Unreachable / Invalid",
                 "Severity": "Informational", "Score": 0,
                 "Recommendation": "Verify target URL."}]
    try:
        root = ET.parse(xml_file).getroot()
        for host in root.findall("host"):
            for port in host.findall(".//port"):
                svc          = port.find("service")
                service_name = svc.get("name") if svc is not None else "unknown"
                port_id      = port.get("portid")

                # Vuln Type 1, Cleartext Protocol (Critical)
                if port_id in ["21", "23"] or service_name in ["ftp", "telnet"]:
                    findings.append({
                        "Target": target,
                        "Vulnerability": f"Cleartext Protocol Exposed ({service_name.upper()})",
                        "Severity": "Critical", "Score": 9,
                        "Recommendation": "Disable service; use secure alternatives like SFTP or SSH."
                    })

                # Vuln Type 2, Remote Admin Interface (High)
                elif port_id in ["22", "3389"] or service_name in ["ssh", "ms-wbt-server"]:
                    findings.append({
                        "Target": target,
                        "Vulnerability": f"Remote Admin Interface Exposed ({service_name.upper()})",
                        "Severity": "High", "Score": 7,
                        "Recommendation": "Restrict access via firewall to trusted IPs or VPN only."
                    })

                # Vuln Type 3, Database Exposed to Public (High)
                elif port_id in ["3306", "5432", "1433", "27017"]:
                    findings.append({
                        "Target": target,
                        "Vulnerability": f"Database Exposed to Public (Port {port_id})",
                        "Severity": "High", "Score": 8,
                        "Recommendation": "Bind database to localhost and block external port access."
                    })

                # Vuln Type 4, Unencrypted Web Traffic (Medium)
                elif port_id in ["80", "8080"] or service_name in ["http", "http-proxy"]:
                    findings.append({
                        "Target": target,
                        "Vulnerability": f"Unencrypted Web Traffic (Port {port_id})",
                        "Severity": "Medium", "Score": 4,
                        "Recommendation": "Enforce HTTPS/TLS and redirect port 80 traffic."
                    })

                # Vuln Type 5, Generic Open Port (Low)
                else:
                    findings.append({
                        "Target": target,
                        "Vulnerability": f"Open Port {port_id} ({service_name})",
                        "Severity": "Low", "Score": 2,
                        "Recommendation": "Ensure service is required, patched, and securely configured."
                    })
    except Exception:
        pass   # Malformed XML or parse error, returns empty list gracefully
    return findings

# Vuln Type 6, VirusTotal Malicious Reputation (Critical) 
def check_virustotal(target, api_key):
    if not api_key:
        return []
    try:
        r = requests.get(
            f"https://www.virustotal.com/api/v3/domains/{target}",
            headers={"x-apikey": api_key}, timeout=5
        )
        stats     = r.json().get("data", {}).get("attributes", {}).get("last_analysis_stats", {})
        malicious = stats.get("malicious", 0)
        if malicious > 0:
            return [{
                "Target": target,
                "Vulnerability": f"Malicious Reputation ({malicious} VT engines flagged)",
                "Severity": "Critical", "Score": 10,
                "Recommendation": "Investigate immediately for compromise and rotate IPs/Domains."
            }]
    except Exception:
        pass
    return []

print("Requirement 1 — Scanning functions defined successfully.")

Requirement 1 — Scanning functions defined successfully.


---
## Requirement 2: Risk Evaluation Dashboard
> Design and build a dashboard to visualize scan results.  
> • Original layout, different from anything shown in class  
> • Must help a viewer quickly understand the risk status of the scanned website

---

### Solution:

A **Streamlit** dashboard with a dark pirate-themed UI (`#0d1117` background, gold headings, scarlet accents).

**Dashboard components:**

| Widget | Purpose |
|---|---|
| 4 KPI metric cards | At-a-glance counts: Summary of scan, Observed Trends, Threat Intelligence, Export|
| Styled DataFrame | Full findings table with colour-coded severity |
| Donut pie chart | Overall distribution of vulnerability severities |
| Stacked bar chart | Per-target vulnerability volume broken down by severity |
| Line chart | Max risk score per target for visual threat ranking |
| Horizontal bar chart | Top 5 most frequent vulnerability types across all targets |
| Alert cards | Top 5 Critical/High/Medium findings with recommendations |



In [4]:
# REQUIREMENT 2 SOLUTION (Risk Evaluation Dashboard)
dashboard_ui_preview = '''
# ── Theme & Page Config ───────────────────────────────────────────────────────
st.set_page_config(page_title="Threat Scanner", page_icon="🏴\u200d☠️", layout="wide")
st.markdown("""
    <style>
    .stApp { background-color: #0d1117; color: #c9d1d9; }
    h1, h2, h3 { color: #FFD700 !important; font-family: \'Courier New\', Courier, monospace; }
    .stButton>button { background-color: #FF2400; color: white;
                       border: 2px solid #FFD700; border-radius: 8px; font-weight: bold; }
    </style>
""", unsafe_allow_html=True)

# ── KPI Metric Cards (4 columns) ─────────────────────────────────────────────
c1, c2, c3, c4 = st.columns(4)
c1.metric("🎯 Targets Scanned",       df["Target"].nunique())
c2.metric("⛓️ Total Vulnerabilities", len(df))
c3.metric("⚠️ Critical/High Threats", len(df[df["Severity"].isin(["Critical", "High"])]))
c4.metric("🏴\u200d☠️ Max Risk Score",    df["Score"].max())

# ── Colour-coded Findings Table ───────────────────────────────────────────────
def color_severity(val):
    colors = {
        \'Critical\':      \'color: #FF0000; font-weight: bold;\',
        \'High\'    :      \'color: #FF4500;\',
        \'Medium\'  :      \'color: #FFD700;\',
        \'Low\'     :      \'color: #1E90FF;\',
        \'Informational\': \'color: #A9A9A9;\'
    }
    return colors.get(val, \'\')
styled_df = df.style.map(color_severity, subset=[\'Severity\'])
st.dataframe(styled_df, use_container_width=True, hide_index=True)

# ── Chart 1: Donut Pie — Severity Distribution ───────────────────────────────
fig_pie = px.pie(sev_counts, names="Severity", values="Count", hole=0.4,
                 title="Overall Vulnerability Distribution",
                 color_discrete_map={"Critical": "#8B0000", "High": "#FF2400",
                                     "Medium": "#FFD700", "Low": "#005A9C",
                                     "Informational": "#808080"})

# ── Chart 2: Stacked Bar — Vulnerabilities per Target ────────────────────────
fig_bar = px.bar(target_counts, x="Target", y="Count", color="Severity",
                 title="Volume of Vulnerabilities by Target")

# ── Chart 3: Line Chart — Max Risk Score per Target ──────────────────────────
fig_line = px.line(max_score_df, x="Target", y="Score", markers=True,
                   title="Target vs Score",
                   color_discrete_sequence=["#FFD700"])

# ── Chart 4: Horizontal Bar — Top 5 Most Frequent Vulnerabilities ─────────────
fig_hbar = px.bar(vuln_counts.head(5), x="Count", y="Vulnerability",
                  orientation=\'h\', title="Top 5 Frequent Issues",
                  color="Count", color_continuous_scale="Reds")

# ── Alert Cards — Top 5 High/Critical/Medium Findings ────────────────────────
for idx, row in top_vulns.head(5).iterrows():
    st.error(f"**{row[\'Target\']}** - {row[\'Vulnerability\']} (Score: {row[\'Score\']})"
             f"  \\n*Recommendation:* {row[\'Recommendation\']}")
'''

print("Requirement 2 — Dashboard UI logic preview displayed.")
print("Full implementation is written to disk in the launch cell (see below).")

Requirement 2 — Dashboard UI logic preview displayed.
Full implementation is written to disk in the launch cell (see below).


---
## Requirement 3: Email Alert System
> Build an alert module that automatically sends an HTML-formatted email when a High or Critical vulnerability is detected.  
> • Subject line indicating severity level and target website  
> • Summary table of all High and Critical findings  
> • Target URL, scan timestamp, and overall risk score  
> • Recommended action for each finding  
> • Automated disclaimer in the footer  
> • Must trigger automatically — not manually  
> • Medium, Low, Informational must NOT trigger an alert

---

### Solution:

The `send_automated_alert()` function handles this entirely.  

**Automatic trigger (not manual):** It is called directly in the post-scan pipeline — only when `Severity` is `High` or `Critical`:
```python
alert_df = df[df["Severity"].isin(["High", "Critical"])]
if not alert_df.empty:
    send_automated_alert(alert_df, targets, st.session_state.scan_time)
```

**Email checklist:**

| Requirement | Implementation |
|---|---|
| Subject with severity + target | `[CRITICAL] Vulnerability Alert for {target}` |
| Summary table (name, severity, score) | HTML `<table>` with all High/Critical rows |
| Target URL + Scan timestamp | Included as bold paragraphs above the table |
| Overall risk score | `df["Score"].max()` shown in red |
| Recommended action per finding | `Recommendation` column in the table |
| Automated disclaimer in footer | `<em>Automated Disclaimer: ...</em>` below a `<hr>` |
| Credentials from environment | `os.environ.get()` — never hardcoded |

In [5]:
# REQUIREMENT 3 SOLUTION - (Automated Email Alert System)
import smtplib
import os
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

# Credentials loaded from environment variables 
sender_email   = os.environ.get("GMAIL_SENDER")
app_password   = os.environ.get("GMAIL_PASSWORD")
recipient_email = None  #user must input

def send_automated_alert(high_crit_df, target_summary, timestamp):
    """Sends an HTML alert email for High/Critical findings.
    Triggered automatically after scanning — no manual action required.
    Only fires if High or Critical findings exist.
    """
    if not sender_email or not app_password or not recipient_email:
        return  # Graceful exit if credentials are not configured

    overall_risk     = high_crit_df["Score"].max()
    subject_severity = "CRITICAL" if overall_risk >= 9 else "HIGH"   # Subject severity
    targets_str      = ", ".join(high_crit_df["Target"].unique())     # Target name(s)

    # HTML Email Body 
    html = f"""
    <html>
      <body style="font-family: Arial, sans-serif; color: #333;">
        <h2 style="color: #FF2400;">🚨 Threat Scanner - Security Alert</h2>

        <!-- Target URL + Scan Timestamp + Overall Risk Score -->
        <p><strong>Scan Timestamp:</strong> {timestamp}</p>
        <p><strong>Target(s):</strong> {targets_str}</p>
        <p><strong>Overall Risk Score:</strong>
           <span style="color: red; font-size: 1.2em;">{overall_risk}/10</span></p>

        <!-- Summary Table: name, severity, score, recommended action -->
        <h3>High &amp; Critical Findings Summary</h3>
        <table border="1" cellpadding="8" style="border-collapse: collapse; width: 100%;">
          <tr style="background-color: #f2f2f2;">
            <th>Target</th>
            <th>Vulnerability</th>
            <th>Severity</th>
            <th>Score</th>
            <th>Recommended Action</th>
          </tr>
    """
    for _, row in high_crit_df.iterrows():
        colour = 'red' if row['Severity'] == 'Critical' else 'orange'
        html += f"""
          <tr>
            <td>{row['Target']}</td>
            <td>{row['Vulnerability']}</td>
            <td style="color: {colour};"><b>{row['Severity']}</b></td>
            <td>{row['Score']}</td>
            <td>{row['Recommendation']}</td>
          </tr>
        """
    html += """
        </table>
        <br><hr>
        <!-- Automated disclaimer in footer -->
        <p style="font-size: 0.8em; color: #777;">
          <em>Automated Disclaimer: This email was generated automatically by the
          Threat Scanner system. Do not reply directly to this message.
          Information contained herein is for authorized defensive purposes only.</em>
        </p>
      </body>
    </html>
    """

    #  Send via Gmail SMTP 
    msg            = MIMEMultipart("alternative")
    msg["Subject"] = f"[{subject_severity}] Vulnerability Alert for {targets_str}"
    msg["From"]    = sender_email
    msg["To"]      = recipient_email
    msg.attach(MIMEText(html, "html"))

    try:
        server = smtplib.SMTP("smtp.gmail.com", 587)
        server.starttls()
        server.login(sender_email, app_password)
        server.send_message(msg)
        server.quit()
        print("Email alert sent successfully.")
    except Exception as e:
        print(f"Failed to send email alert: {e}")

print("Requirement 3 - Email alert function defined successfully.")

Requirement 3 - Email alert function defined successfully.


---
## Setup & Launch
Run the cells below **in order** to install dependencies, write the app to disk, and launch it via a Cloudflare tunnel (Google Colab).

In [6]:
# Cell A: Installing Requirements (google colab)
!apt-get install -y -qq nmap
!pip install streamlit plotly -q
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

'apt-get' is not recognized as an internal or external command,
operable program or batch file.
'wget' is not recognized as an internal or external command,
operable program or batch file.
'dpkg' is not recognized as an internal or external command,
operable program or batch file.


In [ ]:
#STREAMLIT DASHBOARD

%%writefile dashboard.py
import streamlit as st
import subprocess
import xml.etree.ElementTree as ET
import requests
import pandas as pd
import plotly.express as px
import time
import os
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from datetime import datetime

sender_email = os.environ.get("GMAIL_SENDER")
app_password = os.environ.get("GMAIL_PASSWORD")


st.set_page_config(page_title="Threat Scanner", page_icon="🏴‍☠️", layout="wide")

st.markdown("""
    <style>
    .stApp { background-color: #0d1117; color: #c9d1d9; }
    h1, h2, h3 { color: #FFD700 !important; font-family: 'Courier New', Courier, monospace; }
    .stButton>button { background-color: #FF2400; color: white; border: 2px solid #FFD700; border-radius: 8px; font-weight: bold;}
    .stButton>button:hover { background-color: #8B0000; color: #FFD700; border: 2px solid white; }
    .stAlert { background-color: #161b22; border-left: 5px solid #005A9C; }
    hr { border-color: #FF2400; }
    </style>
""", unsafe_allow_html=True)

st.title("🏴‍☠️ Threat Scanner")
st.caption("Network health & Threat Analysis Dashboard")
st.divider()

st.sidebar.title("⚙️ Settings")
st.sidebar.caption("Configure your scanner.")

target_input = st.sidebar.text_area("Enter Targets (comma or newline separated)", value="")
targets = [t.strip() for t in target_input.replace('\n', ',').split(',') if t.strip()]

st.sidebar.divider()
st.sidebar.subheader("🔑 API Key")
VT_API_KEY = st.sidebar.text_input("Enter VirusTotal API Key", type="password")

st.sidebar.divider()
st.sidebar.subheader("✉️ Email Setup")
recipient_email = st.sidebar.text_input("Alert Recipient Address")

st.sidebar.divider()
st.sidebar.subheader("📊 Status")
st.sidebar.write(f"**Targets Ready:** {len(targets)}")

if VT_API_KEY:
    st.sidebar.success("VT API Key: Loaded")
else:
    st.sidebar.warning("VT API Key: Missing 🔴")

if sender_email and app_password:
    st.sidebar.success("Sender Credentials: Loaded")
else:
    st.sidebar.error("Sender Credentials: Missing 🔴")

if sender_email and app_password and recipient_email:
    st.sidebar.success("Alert System: Configured")
else:
    st.sidebar.warning("Alert System: Yet to be Configured 🔴")

run_scan = st.sidebar.button("🔍 Run Scan", use_container_width=True)


SCAN_DIR = "scan_results"
os.makedirs(SCAN_DIR, exist_ok=True)

def run_nmap_scan(target):
    xml_file = f"{SCAN_DIR}/{target}.xml"
    try:
        subprocess.run(["nmap", "-Pn", "-sV", "-oX", xml_file, target], capture_output=True, timeout=120)
    except Exception as e:
        pass
    return xml_file

def parse_nmap(xml_file, target):
    findings = []
    if not os.path.exists(xml_file):
        return [{"Target": target, "Vulnerability": "Host Unreachable / Invalid", "Severity": "Informational", "Score": 0, "Recommendation": "Verify target URL."}]

    try:
        root = ET.parse(xml_file).getroot()
        for host in root.findall("host"):
            for port in host.findall(".//port"):
                svc = port.find("service")
                service_name = svc.get("name") if svc is not None else "unknown"
                port_id = port.get("portid")

                if port_id in ["21", "23"] or service_name in ["ftp", "telnet"]:
                    findings.append({"Target": target, "Vulnerability": f"Cleartext Protocol Exposed ({service_name.upper()})", "Severity": "Critical", "Score": 9, "Recommendation": "Disable service; use secure alternatives like SFTP or SSH."})
                elif port_id in ["22", "3389"] or service_name in ["ssh", "ms-wbt-server"]:
                    findings.append({"Target": target, "Vulnerability": f"Remote Admin Interface Exposed ({service_name.upper()})", "Severity": "High", "Score": 7, "Recommendation": "Restrict access via firewall to trusted IPs or VPN only."})
                elif port_id in ["3306", "5432", "1433", "27017"]:
                    findings.append({"Target": target, "Vulnerability": f"Database Exposed to Public (Port {port_id})", "Severity": "High", "Score": 8, "Recommendation": "Bind database to localhost and block external port access."})
                elif port_id in ["80", "8080"] or service_name in ["http", "http-proxy"]:
                    findings.append({"Target": target, "Vulnerability": f"Unencrypted Web Traffic (Port {port_id})", "Severity": "Medium", "Score": 4, "Recommendation": "Enforce HTTPS/TLS and redirect port 80 traffic."})
                else:
                    findings.append({"Target": target, "Vulnerability": f"Open Port {port_id} ({service_name})", "Severity": "Low", "Score": 2, "Recommendation": "Ensure service is required, patched, and securely configured."})
    except Exception:
        pass
    return findings

def check_virustotal(target, api_key):
    if not api_key: return []
    try:
        r = requests.get(f"https://www.virustotal.com/api/v3/domains/{target}", headers={"x-apikey": api_key}, timeout=5)
        stats = r.json().get("data", {}).get("attributes", {}).get("last_analysis_stats", {})
        malicious = stats.get("malicious", 0)
        if malicious > 0:
            return [{"Target": target, "Vulnerability": f"Malicious Reputation ({malicious} VT engines flagged)", "Severity": "Critical", "Score": 10, "Recommendation": "Investigate immediately for compromise and rotate IPs/Domains."}]
    except Exception:
        pass
    return []


def send_automated_alert(high_crit_df, target_summary, timestamp):
    if not sender_email or not app_password or not recipient_email:
        return

    overall_risk = high_crit_df["Score"].max()
    subject_severity = "CRITICAL" if overall_risk >= 9 else "HIGH"
    targets_str = ", ".join(high_crit_df["Target"].unique())

    html = f"""
    <html>
      <body style="font-family: Arial, sans-serif; color: #333;">
        <h2 style="color: #FF2400;">🚨Threat Scanner - Security Alert</h2>
        <p><strong>Scan Timestamp:</strong> {timestamp}</p>
        <p><strong>Target(s):</strong> {targets_str}</p>
        <p><strong>Overall Risk Score:</strong> <span style="color: red; font-size: 1.2em;">{overall_risk}/10</span></p>

        <h3>High & Critical Findings Summary</h3>
        <table border="1" cellpadding="8" style="border-collapse: collapse; width: 100%;">
          <tr style="background-color: #f2f2f2;">
            <th>Target</th><th>Vulnerability</th><th>Severity</th><th>Score</th><th>Recommended Action</th>
          </tr>
    """
    for _, row in high_crit_df.iterrows():
        html += f"""
          <tr>
            <td>{row['Target']}</td><td>{row['Vulnerability']}</td>
            <td style="color: {'red' if row['Severity']=='Critical' else 'orange'};"><b>{row['Severity']}</b></td>
            <td>{row['Score']}</td><td>{row['Recommendation']}</td>
          </tr>
        """
    html += """
        </table>
        <br>
        <hr>
        <p style="font-size: 0.8em; color: #777;">
          <em>Automated Disclaimer: This email was generated automatically by the Threat Scanner system.
          Do not reply directly to this message. Information contained herein is for authorized defensive purposes only.</em>
        </p>
      </body>
    </html>
    """

    msg = MIMEMultipart("alternative")
    msg["Subject"] = f"[{subject_severity}] Vulnerability Alert for {targets_str}"
    msg["From"] = sender_email
    msg["To"] = recipient_email
    msg.attach(MIMEText(html, "html"))

    try:
        server = smtplib.SMTP("smtp.gmail.com", 587)
        server.starttls()
        server.login(sender_email, app_password)
        server.send_message(msg)
        server.quit()
        st.toast("📧 Automated alert sent successfully!")
    except Exception as e:
        st.error(f"Failed to send email alert: {e}")

if "results_df" not in st.session_state:
    st.session_state.results_df = pd.DataFrame()

if run_scan:
    if not targets:
        st.error("❌ Please enter at least one target in the sidebar.")
    else:
        all_findings = []
        progress_text = st.empty()
        bar = st.progress(0)

        for i, target in enumerate(targets):
            progress_text.text(f"Scanning {target}... (Ports, Services, Reputation)")

            nmap_xml = run_nmap_scan(target)
            all_findings.extend(parse_nmap(nmap_xml, target))
            all_findings.extend(check_virustotal(target, VT_API_KEY))

            bar.progress((i + 1) / len(targets))

        st.session_state.results_df = pd.DataFrame(all_findings)
        st.session_state.scan_time = datetime.now().strftime("%d %b %Y %H:%M:%S")
        progress_text.empty()
        bar.empty()

        df = st.session_state.results_df
        if not df.empty:
            alert_df = df[df["Severity"].isin(["High", "Critical"])]
            if not alert_df.empty:
                send_automated_alert(alert_df, targets, st.session_state.scan_time)


df = st.session_state.results_df

if df.empty:
    st.info("It looks calm here. Enter targets and run the scan to see results.")
else:
    st.success(f"Scan Completed: {st.session_state.scan_time}")

    c1, c2, c3, c4 = st.columns(4)
    c1.metric("🎯Targets Scanned", df["Target"].nunique())
    c2.metric("⛓️‍💥Total Vulnerabilities", len(df))
    c3.metric("⚠️Critical/High Threats", len(df[df["Severity"].isin(["Critical", "High"])]))
    c4.metric("🏴‍☠️Max Risk Score", df["Score"].max())
    st.divider()

    # ── TABS ──────────────────────────────────────────────────────────────────
    tab1, tab2, tab3, tab4 = st.tabs([
        "📜 Summary of Scan",
        "📈 Observed Trends",
        "🚨 Threat Intelligence",
        "💾 Export"
    ])

    # ── TAB 1: SUMMARY OF SCAN ────────────────────────────────────────────────
    with tab1:
        st.subheader("📜 Summary of the Scan")

        def color_severity(val):
            colors = {'Critical': 'color: #FF0000; font-weight: bold;', 'High': 'color: #FF4500;',
                      'Medium': 'color: #FFD700;', 'Low': 'color: #1E90FF;', 'Informational': 'color: #A9A9A9;'}
            return colors.get(val, '')

        styled_df = df.style.map(color_severity, subset=['Severity'])
        st.dataframe(styled_df, use_container_width=True, hide_index=True)

    # ── TAB 2: OBSERVED TRENDS ────────────────────────────────────────────────
    with tab2:
        st.subheader("📈 Observed Trends")
        ch1, ch2 = st.columns(2)

        with ch1:
            sev_counts = df["Severity"].value_counts().reset_index()
            sev_counts.columns = ["Severity", "Count"]
            fig_pie = px.pie(sev_counts, names="Severity", values="Count", hole=0.4,
                             title="Overall Vulnerability Distribution",
                             color="Severity",
                             color_discrete_map={"Critical": "#8B0000", "High": "#FF2400", "Medium": "#FFD700", "Low": "#005A9C", "Informational": "#808080"})
            fig_pie.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)", font_color="white")
            st.plotly_chart(fig_pie, use_container_width=True)

        with ch2:
            target_counts = df.groupby(["Target", "Severity"]).size().reset_index(name="Count")
            fig_bar = px.bar(target_counts, x="Target", y="Count", color="Severity",
                             title="Volume of Vulnerabilities by Target",
                             color_discrete_map={"Critical": "#8B0000", "High": "#FF2400", "Medium": "#FFD700", "Low": "#005A9C", "Informational": "#808080"})
            fig_bar.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)", font_color="white")
            st.plotly_chart(fig_bar, use_container_width=True)

        ch3, ch4 = st.columns(2)

        with ch3:
            max_score_df = df.groupby("Target")["Score"].max().reset_index()
            fig_line = px.line(max_score_df, x="Target", y="Score", markers=True,
                               title="Target vs Score",
                               color_discrete_sequence=["#FFD700"])
            fig_line.update_traces(marker=dict(size=12, color="#FF2400", line=dict(width=2, color="white")))
            fig_line.update_layout(
                paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)", font_color="white",
                yaxis=dict(range=[0, 11], title="Max Risk Score"),
                xaxis=dict(title="Target")
            )
            st.plotly_chart(fig_line, use_container_width=True)

        with ch4:
            vuln_counts = df["Vulnerability"].value_counts().reset_index()
            vuln_counts.columns = ["Vulnerability", "Count"]
            fig_hbar = px.bar(vuln_counts.head(5), x="Count", y="Vulnerability", orientation='h',
                              title="Top 5 Frequent Issues",
                              color="Count", color_continuous_scale="Reds")
            fig_hbar.update_layout(
                paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)", font_color="white",
                yaxis={'categoryorder':'total ascending'}
            )
            st.plotly_chart(fig_hbar, use_container_width=True)

    # ── TAB 3: THREAT INTELLIGENCE ────────────────────────────────────────────
    with tab3:
        st.subheader("⚠️ Top Vulnerabilities")

        high_crit = df[df["Severity"].isin(["Critical", "High"])].sort_values(by="Score", ascending=False)
        medium    = df[df["Severity"] == "Medium"].sort_values(by="Score", ascending=False)

        if high_crit.empty:
            st.success("✅ No Critical or High threats found. Your ship is safe!")
        else:
            st.error(f"🚨 {len(high_crit)} Critical/High finding(s) detected across {high_crit['Target'].nunique()} target(s). Immediate action required.")

        with st.expander(f"🔴 Critical & High Findings ({len(high_crit)})", expanded=True):
            if high_crit.empty:
                st.write("None found.")
            else:
                for idx, row in high_crit.iterrows():
                    st.error(f"**{row['Target']}** — {row['Vulnerability']} (Score: {row['Score']})  \n*Recommendation:* {row['Recommendation']}")

        with st.expander(f"🟡 Medium Findings ({len(medium)})", expanded=False):
            if medium.empty:
                st.write("None found.")
            else:
                for idx, row in medium.iterrows():
                    st.warning(f"**{row['Target']}** — {row['Vulnerability']} (Score: {row['Score']})  \n*Recommendation:* {row['Recommendation']}")

        st.divider()
        st.subheader("🎯 Per-Target Risk Breakdown")
        breakdown = df.groupby("Target").agg(
            Total_Findings      = ("Vulnerability", "count"),
            Critical_High       = ("Severity",      lambda x: x.isin(["Critical", "High"]).sum()),
            Max_Score           = ("Score",          "max"),
            Top_Vulnerability   = ("Vulnerability",  lambda x: x.iloc[0])
        ).reset_index()
        breakdown.columns = ["Target", "Total Findings", "Critical/High Count", "Max Score", "Top Vulnerability"]
        breakdown = breakdown.sort_values("Max Score", ascending=False)
        st.dataframe(breakdown, use_container_width=True, hide_index=True)

    # ── TAB 4: EXPORT ─────────────────────────────────────────────────────────
    with tab4:
        st.subheader("💾 Export Results")

        ea, eb = st.columns(2)
        with ea:
            st.markdown("**Full Scan Results**")
            st.caption("All findings from the scan")
            st.download_button(
                label="⬇️ Download Full Results (CSV)",
                data=df.to_csv(index=False).encode("utf-8"),
                file_name="full_scan_results.csv",
                mime="text/csv",
                use_container_width=True
            )
        with eb:
            st.markdown("**Critical & High Only**")
            st.caption("Filtered to Critical and High severity findings")
            crit_high_df = df[df["Severity"].isin(["Critical", "High"])]
            st.download_button(
                label="⬇️ Download Critical/High (CSV)",
                data=crit_high_df.to_csv(index=False).encode("utf-8"),
                file_name="critical_high_findings.csv",
                mime="text/csv",
                use_container_width=True
            )

        st.divider()
        st.markdown("**Per-Target Summary Report**")
        st.caption("One row per target with aggregated stats")
        summary_export = df.groupby("Target").agg(
            Total_Findings = ("Vulnerability", "count"),
            Critical_High  = ("Severity",      lambda x: x.isin(["Critical", "High"]).sum()),
            Max_Score      = ("Score",          "max"),
            Severities     = ("Severity",       lambda x: ", ".join(sorted(x.unique())))
        ).reset_index()
        summary_export.columns = ["Target", "Total Findings", "Critical/High Count", "Max Score", "Severities Found"]
        st.dataframe(summary_export, use_container_width=True, hide_index=True)
        st.download_button(
            label="⬇️ Download Target Summary (CSV)",
            data=summary_export.to_csv(index=False).encode("utf-8"),
            file_name="target_summary.csv",
            mime="text/csv"
        )

Writing dashboard.py


In [8]:
#Cell C: Write Streamlit Theme Config 
import os
os.makedirs('.streamlit', exist_ok=True)

config = '''
[theme]
primaryColor             = "#FF2400"
backgroundColor          = "#0d1117"
secondaryBackgroundColor = "#161b22"
textColor                = "#c9d1d9"
font                     = "monospace"
'''
open('.streamlit/config.toml', 'w').write(config)
print('.streamlit/config.toml written ✅')
print('Now run Cell D (Streamlit) and Cell E (Cloudflare tunnel) below.')

.streamlit/config.toml written ✅
Now run Cell D (Streamlit) and Cell E (Cloudflare tunnel) below.


In [9]:
#Cell D: Launch Streamlit (Google Colab) 
# Reads API keys from Colab Secrets and passes them as environment variables.
import subprocess, threading, time, os
from google.colab import userdata

def run_streamlit():
    api_key        = userdata.get("API_KEY")
    gmail_sender   = userdata.get("GMAIL_SENDER")
    gmail_password = userdata.get("GMAIL_PASSWORD")

    env_vars = os.environ.copy()
    if api_key:        env_vars["API_KEY"]        = api_key
    if gmail_sender:   env_vars["GMAIL_SENDER"]   = gmail_sender
    if gmail_password: env_vars["GMAIL_PASSWORD"] = gmail_password

    subprocess.run(
        ['streamlit', 'run', 'dashboard.py', '--server.port', '8501', '--server.headless', 'true'],
        env=env_vars
    )

threading.Thread(target=run_streamlit, daemon=True).start()
time.sleep(5)
print("Streamlit is running on port 8501")

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# Cell E: Cloudflare Tunnel, generates a public URL 
# Copy the trycloudflare.com link from the output and open it in your browser.
!cloudflared tunnel --url http://localhost:8501

2026-03-19T08:18:03Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-03-19T08:18:03Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-03-19T08:18:08Z INF +--------------------------------------------------------------------------------------------+
2026-03-19T08:18:08Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-03-19T08:18:08Z INF |  https://listen-soldiers-offer-fill.trycloudflare.com 

In [ ]:
!curl -s http://localhost:8501